Recursively find all possible text files (txt, tsv, csv, excel) and parquet

Firstly clean up all txt files, then tsv, csv, excel, parquet, html, other

In [ ]:
import pandas as pd

In [ ]:
# Path to the parquet file
file_path = r'c:\Users\zabit\Documents\GitHub\Lithuanian-Word-Square\data\aiana94_polynews\train.parquet.gzip'

# Read the parquet file
df = pd.read_parquet(file_path)

# Display the first few rows to view the contents
df.head(10)["text"]

In [ ]:
import spacy

# Load the Lithuanian model
nlp = spacy.load("lt_core_news_lg")

In [ ]:
df["processed_text"] = df["text"].apply(lambda x: "|".join([token.text for token in nlp(x)]))

In [ ]:
import pandas as pd

# Split strings into lists, explode into rows, drop empties, lowercase, then count
counts = (
    df["processed_text"].dropna()
      .str.split("|")
      .explode()
      .replace("", pd.NA)
      .dropna()
      .str.lower()
      .value_counts()   # returns Series indexed by word, sorted by count desc
)

# Save as TSV with header "count"
output_path = "data/aiana94_polynews/lithuanian_word_counts.tsv"
counts.to_csv(output_path, sep="\t", header=["count"])

print(f"Exported {counts.size} distinct words with counts to {output_path}")

In [ ]:
# Read the file and extract words
with open("data/ccll2_vs_war_in_UA/ccll2_vs_war_in_UA.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()

# Extract the first column (word) from each line, strip whitespace, and filter out empties
words = [line.split('\t')[0].strip() for line in lines if line.strip()]

# Collect unique words (case-sensitive, as in your notebook)
unique_words = set(words)
unique_words.discard("")  # Remove any empty strings if present

# Sort the words (optional, for readability)
sorted_words = sorted(unique_words)

# Export to a new text file (one word per line)
with open("data/ccll2_vs_war_in_UA/words_cleaned.txt", "w", encoding="utf-8") as f:
    for word in sorted_words:
        f.write(word + "\n")

print(f"Exported {len(sorted_words)} unique words to ccll2_vs_war_in_UA/words_cleaned.txt")

In [ ]:
import pandas as pd
import spacy

# Path to the tapaco parquet file
tapaco_file_path = r'data\community-datasets_tapaco\train-00000-of-00001.parquet'

# Read the parquet file
tapaco_df = pd.read_parquet(tapaco_file_path)

# Load the Lithuanian spaCy model (do this once)
nlp = spacy.load("lt_core_news_lg", disable=["parser", "ner", "lemmatizer"])  # tokenization only

# Tokenize in batches and keep only alphabetic tokens
texts = tapaco_df["paraphrase"].fillna("").astype(str)

tokens_list = []
for doc in nlp.pipe(texts, batch_size=50):
    # keep only alphabetic tokens; change/remove this filter if you want punctuation/numbers
    tokens_list.append([token.text for token in doc if token.is_alpha])

# Attach tokens (optional - useful for inspection)
tapaco_df["tokens"] = tokens_list

# Explode tokens into a single Series and compute counts
tokens_series = tapaco_df["tokens"].explode().dropna().astype(str)
counts = tokens_series.str.lower().value_counts()   # lowercased aggregation

# Save counts as TSV with header "count" and index being the word
output_path = "data/community-datasets_tapaco/lithuanian_word_counts.tsv"
counts.to_csv(output_path, sep="\t", header=["count"])

print(f"Exported {counts.size} distinct words with counts to {output_path}")

In [ ]:
import os
from collections import Counter
import spacy

print("Importing libraries...")

# Base directory and subdirectories to process
corpus_base_dir = os.path.join('data', 'Corpus_of_Discourse_on_Crime-LT')
subcorpora = [os.path.join(corpus_base_dir, 'Subcorpus_1'),
              os.path.join(corpus_base_dir, 'Subcorpus_2')]

# Load spaCy Lithuanian tokenizer (disable heavier pipeline components for speed)
nlp = spacy.load("lt_core_news_lg", disable=["parser", "ner", "lemmatizer"])
nlp.max_length = 2000000

# Counter to accumulate token frequencies
counter = Counter()

print("Starting to process files...")
for sub_dir in subcorpora:
    for root, _, files in os.walk(sub_dir):
        for filename in files:
            if filename.lower().endswith('.txt'):
                file_path = os.path.join(root, filename)
                print(f"Processing: {file_path}")
                try:
                    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                        text = f.read()
                        # Use nlp.pipe over a single-item list to keep consistent pipeline use
                        for doc in nlp.pipe([text], batch_size=1):
                            for token in doc:
                                # Count only alphabetic tokens, normalized to lowercase
                                if token.is_alpha:
                                    counter[token.text.lower()] += 1
                except Exception as e:
                    print(f"Error reading or processing file {file_path}: {e}")

# Export unique words (one per line)
unique_words = sorted(counter.keys())
output_words_file = os.path.join(corpus_base_dir, 'Lithuanian_words.txt')
with open(output_words_file, 'w', encoding='utf-8') as f:
    for word in unique_words:
        f.write(word + '\n')

# Export word counts as TSV (word<TAB>count), sorted by descending count
output_counts_file = os.path.join(corpus_base_dir, 'lithuanian_word_counts.tsv')
with open(output_counts_file, 'w', encoding='utf-8') as f:
    f.write('word\tcount\n')
    for word, cnt in counter.most_common():
        f.write(f"{word}\t{cnt}\n")

print(f"\nProcessing complete.")
print(f"Exported {len(unique_words)} unique words to {output_words_file}")
print(f"Exported counts for {len(counter)} words to {output_counts_file}")

In [2]:
import os
from collections import Counter
import re

# Define the directories containing the text files
reliable_dir = r"data\DIGIRES_COVID19_LT\DIGIRES_corpus-v1\reliable"
unreliable_dir = r"data\DIGIRES_COVID19_LT\DIGIRES_corpus-v1\unreliable"

# Initialize a Counter to store word frequencies
word_counter = Counter()

# Function to process text files in a directory
def process_text_files(directory, word_counter):
    if os.path.exists(directory) and os.path.isdir(directory):
        for filename in os.listdir(directory):
            if filename.endswith(".txt"):  # Process only .txt files
                file_path = os.path.join(directory, filename)
                try:
                    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                        for line in f:
                            # Tokenize the line using regex to extract words
                            words = re.findall(r'\b\w+\b', line.lower())  # Extract words and convert to lowercase
                            word_counter.update(words)
                except Exception as e:
                    print(f"Error reading file {file_path}: {e}")
    else:
        print(f"Directory not found: {directory}")

# Process both directories
process_text_files(reliable_dir, word_counter)
process_text_files(unreliable_dir, word_counter)

# Ensure output directory exists
out_dir = os.path.join("data", "DIGIRES_COVID19_LT")
os.makedirs(out_dir, exist_ok=True)

# Save counts as TSV with header "word\tcount"
output_counts_path = os.path.join(out_dir, "lithuanian_word_counts.tsv")
with open(output_counts_path, "w", encoding="utf-8") as f:
    f.write("word\tcount\n")
    for word, count in word_counter.most_common():
        f.write(f"{word}\t{count}\n")

# Save unique words (one per line) — optional
output_words_path = os.path.join(out_dir, "lit_words.txt")
with open(output_words_path, "w", encoding="utf-8") as f:
    for word in sorted(word_counter.keys()):
        f.write(word + "\n")

print(f"Exported {len(word_counter)} distinct words with counts to {output_counts_path}")
print(f"Exported {len(word_counter)} unique words to {output_words_path}")

Exported 30249 distinct words with counts to data\DIGIRES_COVID19_LT\lithuanian_word_counts.tsv
Exported 30249 unique words to data\DIGIRES_COVID19_LT\lit_words.txt


In [ ]:
# Path to the TSV file
file_path = r'Fizinių_asmenų_bankrotas\Fizinių asmenų bankrotas.csv'

# Read the TSV file using pandas
bankrotai_df = pd.read_csv(file_path, sep=',')

# Display the first few rows to view the contents
bankrotai_df.tail()["Asmuo, kuriam pavesta administruoti - Vardas, pavardė"]

import spacy

# Load the Lithuanian model
nlp = spacy.load("lt_core_news_lg")

bankrotai_df["processed_text"] = bankrotai_df["Vardas"].apply(lambda x: "|".join([token.text for token in nlp(x)]))
bankrotai_df["processed_text"] = bankrotai_df["Pavardė"].apply(lambda x: "|".join([token.text for token in nlp(x)]))
bankrotai_df["processed_text"] = bankrotai_df["Asmuo, kuriam pavesta administruoti - Vardas, pavardė"].apply(lambda x: "|".join([token.text for token in nlp(x)]))


# Collect all unique words from processed_text
all_words = set()
for text in bankrotai_df["processed_text"]:
    words = text.split("|")
    all_words.update(words)

# Remove any empty strings if present
all_words.discard("")

# Sort the words (optional, for readability)
sorted_words = sorted(all_words)

# Export to a text file (one word per line)
with open("Fizinių_asmenų_bankrotas/lit_words.txt", "w", encoding="utf-8") as f:
    for word in sorted_words:
        f.write(word + "\n")

print(f"Exported {len(sorted_words)} unique words to lithuanian_words.txt")

In [ ]:
import pandas as pd
import spacy

# Load the Lithuanian model (do this once, outside loops if possible)
nlp = spacy.load("lt_core_news_lg")

# Path to the parquet file
file_path = r'wikimediawikipedia\train-00000-of-00001.parquet'

# Read the parquet file
df = pd.read_parquet(file_path)

# Display the first few rows (optional, for inspection)
print(df.head(10))

# Combine title and text into a single series for processing (adjust if you only want one)
combined_texts = df["title"].fillna("") + " " + df["text"].fillna("")

# Use nlp.pipe for batch processing (faster than apply)
all_words = set()
for doc in nlp.pipe(combined_texts, batch_size=50):  # Adjust batch_size based on memory
    for token in doc:
        all_words.add(token.text)

# Remove empty strings if present
all_words.discard("")

# Sort if needed (optional; skip for large sets to save time/memory)
sorted_words = sorted(all_words)

# Export to a text file (one word per line)
output_path = "wikimediawikipedia/lit_words.txt"
with open(output_path, "w", encoding="utf-8") as f:
    f.writelines(f"{word}\n" for word in sorted_words)

print(f"Exported {len(sorted_words)} unique words to {output_path}")

In [ ]:
import os
import spacy

print("Importing libraries...")

# Define the base directory and subdirectories to process
corpus_base_dir = 'Corpus_of_Discourse_on_Crime-LT'
subcorpora = [os.path.join(corpus_base_dir, 'Subcorpus_1'), os.path.join(corpus_base_dir, 'Subcorpus_2')]
all_words = set()

# Ensure the spacy model is loaded (assuming it's in 'nlp' from a previous cell)
# If not, uncomment the following line:
nlp = spacy.load("lt_core_news_lg")

print("Starting to process files...")
# Walk through the specified subdirectories
for sub_dir in subcorpora:
    for root, _, files in os.walk(sub_dir):
        for filename in files:
            # Process only text files
            if filename.endswith('.txt'):
                file_path = os.path.join(root, filename)
                print(f"Processing: {file_path}")
                try:
                    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                        text = f.read()
                        # Use spacy to tokenize the text
                        doc = nlp(text)
                        # Add each word to the set to ensure uniqueness
                        for token in doc:
                            all_words.add(token.text)
                except Exception as e:
                    print(f"Error reading or processing file {file_path}: {e}")

# Remove any empty strings that might have been added
all_words.discard("")

# Sort the unique words alphabetically
sorted_words = sorted(list(all_words))

# Define the output file path
output_file = os.path.join(corpus_base_dir, 'Lithuanian_words.txt')

# Write the sorted words to the output file, one word per line
with open(output_file, 'w', encoding='utf-8') as f:
    for word in sorted_words:
        f.write(word + '\n')

print(f"\nProcessing complete.")
print(f"Exported {len(sorted_words)} unique words to {output_file}")

In [ ]:
# Read the first file and extract words
with open(r"DML6_vs_JCL\DML6_lemmas_vs_JCL.txt", "r", encoding="utf-8") as f:
    lines1 = f.readlines()

# Extract words from the first file
words1 = [line.split('\t')[0].strip() for line in lines1 if line.strip()]
unique_words = set(words1)

# Read the second file and add its words to the set
with open(r"DML6_vs_JCL\JCL_types_missing_in_DML6_(filtered_list).txt", "r", encoding="utf-8") as f:
    lines2 = f.readlines()

# Extract words from the second file
words2 = [line.split('\t')[0].strip() for line in lines2 if line.strip()]
unique_words.update(words2)

# Read the third file and add its words to the set
with open(r"DML6_vs_JCL\JCL_types_vs_DML6.txt", "r", encoding="utf-8") as f:
    lines3 = f.readlines()

# Extract words from the third file
words3 = [line.split('\t')[0].strip() for line in lines3 if line.strip()]
unique_words.update(words3)

# Remove any empty strings if present
unique_words.discard("")

# Sort the combined list of unique words
sorted_words = sorted(unique_words)

# Export to the text file, overwriting it with the combined list
output_path = r"DML6_vs_JCL\words_cleaned.txt"
with open(output_path, "w", encoding="utf-8") as f:
    for word in sorted_words:
        f.write(word + "\n")

print(f"Exported {len(sorted_words)} unique words from all files to {output_path}")

In [ ]:
import os
from collections import Counter

# Define the directory containing the text files
corpus_dir = r'data\EN-LT_Comparable_Vaccination_corpus\Lithuanian_media_articles_on_vaccination-full-text_corpus'

# Counter to hold word frequencies
word_counts = Counter()

print(f"Starting to process files in: {corpus_dir}")

# Check if the directory exists
if os.path.exists(corpus_dir) and os.path.isdir(corpus_dir):
    # Loop through all files in the specified directory
    for filename in os.listdir(corpus_dir):
        # Check if the file is a text file
        if filename.endswith('.txt'):
            file_path = os.path.join(corpus_dir, filename)
            print(f"Reading file: {file_path}")
            try:
                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    text = f.read()
                    for doc in nlp.pipe([text], batch_size=1):
                        for token in doc:
                            if token.is_alpha:                   # only alphabetic tokens
                                word_counts[token.text.lower()] += 1
            except Exception as e:
                print(f"Could not read file {file_path}: {e}")

    # Prepare outputs
    sorted_words = sorted(word_counts.keys())
    output_dir = 'data\\EN-LT_Comparable_Vaccination_corpus'
    os.makedirs(output_dir, exist_ok=True)
    output_words_path = os.path.join(output_dir, 'combined_words.txt')
    output_counts_path = os.path.join(output_dir, 'combined_word_counts.tsv')

    # Write unique words (one per line) and counts (TSV)
    try:
        with open(output_words_path, 'w', encoding='utf-8') as f:
            for w in sorted_words:
                f.write(w + '\n')

        # Write counts as TSV with header 'word\tcount'
        with open(output_counts_path, 'w', encoding='utf-8') as f:
            f.write('word\tcount\n')
            for w, c in word_counts.most_common():
                f.write(f"{w}\t{c}\n")

        print(f"\nSuccessfully combined files and wrote counts.")
        print(f"Exported {len(sorted_words)} unique words to {output_words_path}")
        print(f"Exported counts for {len(word_counts)} words to {output_counts_path}")
    except Exception as e:
        print(f"Could not write output files: {e}")
else:
    print(f"Error: Directory not found at '{corpus_dir}'")


In [4]:
import os, time
from collections import Counter
import spacy

# Path to the parquet file
file_path = r'data\wikimediawikipedia\train-00000-of-00001.parquet'

# Read the parquet file
df = pd.read_parquet(file_path)

# Load a smaller spaCy model for speed if available
nlp = spacy.load("lt_core_news_sm", disable=["parser","ner","lemmatizer"])
nlp.max_length = 2000000

# Choose a batch size that fits RAM and CPU (increase for fewer Python calls)
ROW_CHUNK = 5000        # outer chunk of rows to pass to pipe
PIPE_BATCH = 500        # spaCy internal batch_size
counter = Counter()

texts = (df["title"].fillna("") + " " + df["text"].fillna("")).astype(str)
total = len(texts)
start = time.time()

for i in range(0, total, ROW_CHUNK):
    chunk = texts.iloc[i:i+ROW_CHUNK].tolist()
    # tokenize with spaCy pipe; set batch_size to PIPE_BATCH
    for doc in nlp.pipe(chunk, batch_size=PIPE_BATCH):
        for token in doc:
            # normalization: lowercase and keep alphabetic tokens only
            if token.is_alpha:
                counter[token.text.lower()] += 1

    # progress log every chunk (adjust frequency if noisy)
    elapsed = time.time() - start
    processed = min(i + ROW_CHUNK, total)
    print(f"Processed {processed}/{total} rows — unique tokens so far: {len(counter)} — elapsed {elapsed:.1f}s")

# ensure output dir exists and write TSV sorted by frequency
out_dir = os.path.join("data", "wikimediawikipedia")
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "lithuanian_word_counts.tsv")
with open(out_path, "w", encoding="utf-8") as f:
    f.write("word\tcount\n")
    for w, c in counter.most_common():
        f.write(f"{w}\t{c}\n")

print(f"Done. Exported {len(counter)} distinct words to {out_path} (total time {time.time()-start:.1f}s)")

Processed 5000/211292 rows — unique tokens so far: 246993 — elapsed 290.7s
Processed 10000/211292 rows — unique tokens so far: 331882 — elapsed 439.2s
Processed 15000/211292 rows — unique tokens so far: 394822 — elapsed 573.4s
Processed 20000/211292 rows — unique tokens so far: 445490 — elapsed 682.5s
Processed 25000/211292 rows — unique tokens so far: 475374 — elapsed 760.3s
Processed 30000/211292 rows — unique tokens so far: 503904 — elapsed 834.2s
Processed 35000/211292 rows — unique tokens so far: 540381 — elapsed 919.8s
Processed 40000/211292 rows — unique tokens so far: 566854 — elapsed 990.0s
Processed 45000/211292 rows — unique tokens so far: 600626 — elapsed 1078.1s
Processed 50000/211292 rows — unique tokens so far: 622997 — elapsed 1152.2s
Processed 55000/211292 rows — unique tokens so far: 650840 — elapsed 1231.0s
Processed 60000/211292 rows — unique tokens so far: 681317 — elapsed 1325.5s
Processed 65000/211292 rows — unique tokens so far: 708081 — elapsed 1411.2s
Processe

In [6]:
import os
from collections import Counter
import spacy

# Load spaCy Lithuanian tokenizer (disable heavier components for speed)
print("Loading spaCy model (this may take a moment)...")
# You can switch to "lt_core_news_sm" if the large model isn't installed/available
nlp = spacy.load("lt_core_news_lg", disable=["parser", "ner", "lemmatizer"])  # tokenization only
nlp.max_length = 2000000

# Define the directory containing the text files
corpus_dir = os.path.join('data', 'EN-LT_Parallel_CS_Corpus-v2', 'LT-TXT')

# Counter to hold word frequencies across all files
word_counts = Counter()

print(f"Starting to process files in: {corpus_dir}")

# Check if the directory exists
if os.path.exists(corpus_dir) and os.path.isdir(corpus_dir):
    # Loop through all files in the specified directory
    for filename in os.listdir(corpus_dir):
        # Check if the file is a text file
        if filename.lower().endswith('.txt'):
            file_path = os.path.join(corpus_dir, filename)
            print(f"Reading file: {file_path}")
            try:
                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    text = f.read()
                    # Tokenize using spaCy and count alphabetic tokens (lowercased)
                    for doc in nlp.pipe([text], batch_size=1):
                        for token in doc:
                            if token.is_alpha:  # only alphabetic tokens
                                word_counts[token.text.lower()] += 1
            except Exception as e:
                print(f"Could not read file {file_path}: {e}")

    # Prepare outputs
    sorted_words = sorted(word_counts.keys())
    output_dir = os.path.join('data', 'EN-LT_Parallel_CS_Corpus-v2')
    os.makedirs(output_dir, exist_ok=True)
    output_words_path = os.path.join(output_dir, 'combined_words.txt')
    output_counts_path = os.path.join(output_dir, 'combined_word_counts.tsv')

    # Write unique words (one per line)
    try:
        with open(output_words_path, 'w', encoding='utf-8') as f:
            for w in sorted_words:
                f.write(w + '\n')

        # Write counts as TSV with header 'word\tcount' sorted by frequency
        with open(output_counts_path, 'w', encoding='utf-8') as f:
            f.write('word\tcount\n')
            for w, c in word_counts.most_common():
                f.write(f"{w}\t{c}\n")

        print(f"\nSuccessfully combined files and wrote counts.")
        print(f"Exported {len(sorted_words)} unique words to {output_words_path}")
        print(f"Exported counts for {len(word_counts)} words to {output_counts_path}")
    except Exception as e:
        print(f"Could not write output files: {e}")
else:
    print(f"Error: Directory not found at '{corpus_dir}'")


Loading spaCy model (this may take a moment)...
Starting to process files in: data\EN-LT_Parallel_CS_Corpus-v2\LT-TXT
Reading file: data\EN-LT_Parallel_CS_Corpus-v2\LT-TXT\EUR-Lex_001-LT.txt
Reading file: data\EN-LT_Parallel_CS_Corpus-v2\LT-TXT\EUR-Lex_002-LT.txt
Reading file: data\EN-LT_Parallel_CS_Corpus-v2\LT-TXT\EUR-Lex_004-LT.txt
Reading file: data\EN-LT_Parallel_CS_Corpus-v2\LT-TXT\EUR-Lex_005-LT.txt
Reading file: data\EN-LT_Parallel_CS_Corpus-v2\LT-TXT\EUR-Lex_006-LT.txt
Reading file: data\EN-LT_Parallel_CS_Corpus-v2\LT-TXT\EUR-Lex_008-LT.txt
Reading file: data\EN-LT_Parallel_CS_Corpus-v2\LT-TXT\EUR-Lex_009-LT.txt
Reading file: data\EN-LT_Parallel_CS_Corpus-v2\LT-TXT\EUR-Lex_012-LT.txt
Reading file: data\EN-LT_Parallel_CS_Corpus-v2\LT-TXT\EUR-Lex_016-LT.txt
Reading file: data\EN-LT_Parallel_CS_Corpus-v2\LT-TXT\EUR-Lex_018-LT.txt
Reading file: data\EN-LT_Parallel_CS_Corpus-v2\LT-TXT\EUR-Lex_019-LT.txt
Reading file: data\EN-LT_Parallel_CS_Corpus-v2\LT-TXT\EUR-Lex_022-LT.txt
Readin

In [ ]:
from bs4 import BeautifulSoup

# Read the HTML file
file_path = r"data\etimologinio_zodyno_duomenu_baze\Lietuvių kalbos etimologinio žodyno duomenų bazė.htm"
with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
    html_content = f.read()

# Parse the HTML and extract text
soup = BeautifulSoup(html_content, "html.parser")
extracted_text = soup.get_text()

# Skip the first 10 lines
lines = extracted_text.split('\n')
skipped_text = '\n'.join(lines[10:])

# Remove all tab spaces
skipped_text = skipped_text.replace('\t', '')

# Optionally, print a preview of the skipped text
print(skipped_text[:1000])  # Print first 1000 characters as a preview

In [ ]:
def extract_words(text):
    # Split the text by commas and strip whitespace from each item
    items = [item.strip() for item in text.split(',')]
    words = []
    
    for item in items:
        # Skip standalone numbers in parentheses (e.g., "(449)")
        if item.startswith('(') and item.endswith(')') and not any(char.isalpha() for char in item):
            continue
        # Skip items starting with '-' (e.g., "-aitis")
        if item.startswith('-'):
            continue
        # Extract the word by splitting on '(' and taking the first part, then strip
        word = item.split('(')[0].strip()
        if word:  # Ensure it's not empty
            words.append(word)
    
    return words

# Example usage with your string
text = "(449), (iš)skrobti (1), -aitis (1), -dėjas (1), -drusti, -drunda, drudo (1), -kakti (1), -kalys (1), -lika (1), -nikti (2), -palaikis (1), -pi (2), -saja (1), -skusti (1), -ytis (1), abdas (1), abeji (1), abejodas (1), abejodumas (1), abelinis (1), abet kas (1)"
extracted_words = extract_words(skipped_text)

# Assuming extracted_words is the list from the function
with open(r"data\etimologinio_zodyno_duomenu_baze\extracted_words.txt", "w", encoding="utf-8") as f:
    for word in extracted_words:
        f.write(word + "\n")

print(f"Saved {len(extracted_words)} words to extracted_words.txt")


In [ ]:
import pandas as pd
# Path to the TSV file
file_path = r'data\grams\1gram\unigrams_frequency.csv'

# Read the TSV file using pandas
df = pd.read_csv(file_path, sep=',')

print(df.head(10))

# Export to a text file (one word per line), skipping NaN values and converting to string
with open("data/grams/1gram/lit_words.txt", "w", encoding="utf-8") as f:
    for word in df["_id"].dropna():
        f.write(str(word) + "\n")

print(f"Exported {len(df['_id'].dropna())} unique words to lithuanian_words.txt")

In [ ]:
import pandas as pd
# Path to the TSV file
file_path = r'data\grams\2gram\bigrams_frequency.csv'

# Read the TSV file using pandas
df = pd.read_csv(file_path, sep=',')

print(df.head(10))

# Collect unique words from the bigrams
unique_words = set()
for bigram in df["id"].dropna():
    words = str(bigram).split()  # Split the bigram into words
    unique_words.update(words)

# Export to a text file (one word per line), skipping NaN values and converting to string
with open("data/grams/2gram/lit_words.txt", "w", encoding="utf-8") as f:
    for word in sorted(unique_words):  # Sort for consistency
        f.write(word + "\n")

print(f"Exported {len(unique_words)} unique words to lithuanian_words.txt")

In [ ]:
import pandas as pd
# Path to the TSV file
file_path = r'data\grams\3gram\trigrams_frequency.csv'

# Read the TSV file using pandas
df = pd.read_csv(file_path, sep=',')

print(df.head(10))

# Collect unique words from the bigrams
unique_words = set()
for bigram in df["_id"].dropna():
    words = str(bigram).split()  # Split the bigram into words
    unique_words.update(words)

# Export to a text file (one word per line), skipping NaN values and converting to string
with open("data/grams/3gram/lit_words.txt", "w", encoding="utf-8") as f:
    for word in sorted(unique_words):  # Sort for consistency
        f.write(word + "\n")

print(f"Exported {len(unique_words)} unique words to lithuanian_words.txt")

In [ ]:
import pandas as pd

# Path to the CSV file
file_path = r'data\grams\4gram\tetragrams_frequency.csv'

# Collect unique words across all chunks
unique_words = set()

# Process in chunks to avoid loading the entire file
for chunk in pd.read_csv(file_path, sep=',', chunksize=100000):  # Adjust chunksize as needed
    print(f"Processing chunk with {len(chunk)} rows...")
    for tetragram in chunk["_id"].dropna():
        words = str(tetragram).split()  # Split the tetragram into words
        unique_words.update(words)

# Export to a text file (one word per line)
with open("data/grams/4gram/lit_words.txt", "w", encoding="utf-8") as f:
    for word in sorted(unique_words):  # Sort for consistency
        f.write(word + "\n")

print(f"Exported {len(unique_words)} unique words to data/grams/4gram/lit_words.txt")

In [ ]:
# Path to the parquet file
file_path = r'data\HuggingFaceFW_finepdfs\000_00000.parquet'

# Read the parquet file
df = pd.read_parquet(file_path)

print(df.head(10)["text"])

In [ ]:
import spacy

# Load the Lithuanian model (adjust as needed, e.g., disable components if not using them)
nlp = spacy.load("lt_core_news_lg", disable=["parser", "ner", "lemmatizer"])  # Disabling saves memory if you only need tokenization

# Increase the max length to handle longer texts (e.g., 2 million characters)
nlp.max_length = 2000000

In [ ]:
# Use this faster approach:
def process_batch(texts):
    processed = []
    for doc in nlp.pipe(texts, batch_size=100):  # Adjust batch_size based on your memory
        processed.append("|".join([token.text for token in doc]))
    return processed

df["processed_text"] = process_batch(df["text"].fillna(""))

In [ ]:
df["processed_text"] = df["processed_text"].str.replace("|", "\n")
print(df["processed_text"].head(10))

# Save the processed text to the specified file
with open("data/HuggingFaceFW_finepdfs/lit_text.txt", "w", encoding="utf-8") as f:
    for text in df["processed_text"]:
        f.write(text + "\n")

In [ ]:
import pandas as pd
#  Path to the parquet file
file_path = r'data\wikimediawikipedia\train-00000-of-00001.parquet'

# Read the parquet file
df = pd.read_parquet(file_path)

print(df.head(10)["text"])

In [ ]:
import spacy

# Load the Lithuanian model (adjust as needed, e.g., disable components if not using them)
nlp = spacy.load("lt_core_news_lg", disable=["parser", "ner", "lemmatizer"])  # Disabling saves memory if you only need tokenization

# Increase the max length to handle longer texts (e.g., 2 million characters)
nlp.max_length = 2000000

In [ ]:
# Use this faster approach:
def process_batch(texts):
    processed = []
    for doc in nlp.pipe(texts, batch_size=100):  # Adjust batch_size based on your memory
        processed.append("|".join([token.text for token in doc]))
    return processed

df["processed_text"] = process_batch(df["text"].fillna(""))

In [ ]:
df["processed_text"] = df["processed_text"].str.replace("|", "\n")
print(df["processed_text"].head(10))

# Save the processed text to the specified file
with open("data/wikimediawikipedia/lit_text.txt", "w", encoding="utf-8") as f:
    for text in df["processed_text"]:
        f.write(text + "\n")

In [8]:
import pandas as pd
from collections import Counter
import os

# Path to the tab-separated file
file_path = r'data\MATAS-v1.0\combined.txt'

# We'll read the file in chunks to handle large files and show progress
chunksize = 100000  # adjust as needed based on memory
counter = Counter()
rows_processed = 0

# Read in chunks; skip malformed lines if present
for chunk in pd.read_csv(file_path, sep='\t', header=None, chunksize=chunksize, encoding='utf-8', iterator=True, on_bad_lines='skip'):
    # Update counts from the first column (index 0)
    counter.update(chunk[0].dropna().astype(str).tolist())
    rows_processed += len(chunk)
    print(f"Processed {rows_processed} rows — unique words so far: {len(counter)}")

# Prepare output directory
out_dir = os.path.join('data', 'MATAS-v1.0')
os.makedirs(out_dir, exist_ok=True)

# Write counts to TSV with header 'word\tcount' (sorted by frequency)
tsv_path = os.path.join(out_dir, 'lithuanian_word_counts.tsv')
with open(tsv_path, 'w', encoding='utf-8') as f:
    f.write('word\tcount\n')
    for word, cnt in counter.most_common():
        f.write(f"{word}\t{cnt}\n")

# Also export unique words (one per line) for compatibility with existing flow
unique_path = os.path.join(out_dir, 'lit_words.txt')
with open(unique_path, 'w', encoding='utf-8') as f:
    for word in sorted(counter.keys()):
        f.write(word + '\n')

print(f"Exported {len(counter)} distinct words with counts to {tsv_path}")
print(f"Exported {len(counter)} unique words to {unique_path}")

Processed 100000 rows — unique words so far: 6
Processed 200000 rows — unique words so far: 18061
Processed 300000 rows — unique words so far: 30021
Processed 400000 rows — unique words so far: 30021
Processed 500000 rows — unique words so far: 42220
Processed 600000 rows — unique words so far: 53145
Processed 700000 rows — unique words so far: 60284
Processed 800000 rows — unique words so far: 66780
Processed 900000 rows — unique words so far: 66780
Processed 1000000 rows — unique words so far: 66780
Processed 1100000 rows — unique words so far: 66780
Processed 1200000 rows — unique words so far: 76276
Processed 1300000 rows — unique words so far: 81954
Processed 1342318 rows — unique words so far: 81954
Exported 81954 distinct words with counts to data\MATAS-v1.0\lithuanian_word_counts.tsv
Exported 81954 unique words to data\MATAS-v1.0\lit_words.txt


In [ ]:
# Path to the tab-separated file
file_path = r'data\Lithuanian-wordlist\dazninis.txt'

# Read the file using pandas
matas_df = pd.read_csv(file_path, sep='\t', header=None)

# Save the 0 column into a set
unique_words = set(matas_df[2])

# Sort the words (optional, for readability)
sorted_words = sorted(unique_words)

# Export to a text file (one word per line)
with open("data/Lithuanian-wordlist/lit_words.txt", "w", encoding="utf-8") as f:
    for word in sorted_words:
        f.write(word + "\n")

print(f"Exported {len(sorted_words)} unique words to lit_words.txt")

In [ ]:
import os

def combine_txt_files(input_dir, output_file):
    """
    Combines all .txt files in the input_dir into a single output_file.

    Args:
        input_dir (str): The path to the directory containing the .txt files.
        output_file (str): The path to the file where the combined content will be written.
    """
    # Define the full paths based on the user's request structure
    # Note: This assumes the script is run from a location where the relative path
    # 'data\MATAS-v1.0\' is valid from the current working directory.
    # On Windows, backslashes might be an issue in string literals; using os.path.join is safer.

    # Construct the full input and output paths
    base_path = os.path.join('data', 'Lithuanian Coreference Corpus')
    input_directory = os.path.join(base_path, 'corpus')
    output_filepath = os.path.join(base_path, output_file)

    print(f"Looking for .txt files in: {input_directory}")
    print(f"Output will be written to: {output_filepath}")

    # Ensure the output directory exists before writing (optional but good practice)
    output_directory = os.path.dirname(output_filepath)
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
        print(f"Created output directory: {output_directory}")


    # Open the output file in write mode ('w'). This will create the file or overwrite it if it exists.
    try:
        with open(output_filepath, 'w', encoding='utf-8') as outfile:
            # Iterate over all files in the specified directory
            for filename in os.listdir(input_directory):
                # Check if the file ends with '.txt'
                if filename.endswith(".txt"):
                    # Construct the full path to the current input file
                    input_filepath = os.path.join(input_directory, filename)

                    print(f"  Processing: {filename}")

                    # Open the input file in read mode ('r')
                    try:
                        with open(input_filepath, 'r', encoding='utf-8') as infile:
                            # Read the content and write it to the output file
                            content = infile.read()
                            outfile.write(content)

                            # Add a separator (e.g., a newline) between files for clarity,
                            # unless you explicitly want them concatenated without any break.
                            # Adding two newlines ensures there's at least one blank line between files.
                            outfile.write('\n\n')

                    except IOError as e:
                        print(f"Error reading file {filename}: {e}")

        print("\nFile combination complete!")

    except IOError as e:
        print(f"Error writing to output file {output_filepath}: {e}")


# --- Execution ---
if __name__ == "__main__":
    INPUT_DIRECTORY_NAME = "corpus"
    OUTPUT_FILE_NAME = "combined.txt"

    # The base path structure from the request: data\MATAS-v1.0\
    # The function will construct the full paths internally.
    combine_txt_files(INPUT_DIRECTORY_NAME, OUTPUT_FILE_NAME)

In [ ]:
import os

def combine_txt_files(input_dir, output_file):

    # Construct the full input and output paths
    base_path = os.path.join('data', 'KLASIUS', 'VIDURINIO UGDYMO MOKSLEIVIAI')
    input_directory = os.path.join(base_path, input_dir)
    output_filepath = os.path.join(base_path, output_file)

    print(f"Looking for .txt files in: {input_directory}")
    print(f"Output will be written to: {output_filepath}")

    # Ensure the output directory exists before writing (optional but good practice)
    output_directory = os.path.dirname(output_filepath)
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
        print(f"Created output directory: {output_directory}")


    # Open the output file in write mode ('w'). This will create the file or overwrite it if it exists.
    try:
        with open(output_filepath, 'w', encoding='utf-8') as outfile:
            # Iterate over all files in the specified directory
            for filename in os.listdir(input_directory):
                # Check if the file ends with '.txt'
                if filename.endswith(".txt"):
                    # Construct the full path to the current input file
                    input_filepath = os.path.join(input_directory, filename)

                    print(f"  Processing: {filename}")

                    # Open the input file in read mode ('r')
                    try:
                        with open(input_filepath, 'r', encoding='utf-8') as infile:
                            # Read the content and write it to the output file
                            content = infile.read()
                            outfile.write(content)

                            # Add a separator (e.g., a newline) between files for clarity,
                            # unless you explicitly want them concatenated without any break.
                            # Adding two newlines ensures there's at least one blank line between files.
                            outfile.write('\n\n')

                    except IOError as e:
                        print(f"Error reading file {filename}: {e}")

        print("\nFile combination complete!")

    except IOError as e:
        print(f"Error writing to output file {output_filepath}: {e}")


# --- Execution ---
if __name__ == "__main__":
    INPUT_DIRECTORY_NAME = "ye"
    OUTPUT_FILE_NAME = "combined_full.txt"

    # The base path structure from the request: data\MATAS-v1.0\
    # The function will construct the full paths internally.
    combine_txt_files(INPUT_DIRECTORY_NAME, OUTPUT_FILE_NAME)

In [ ]:
# Path to the text file
file_path = r'data\European Parliament Proceedings Parallel Corpus 1996-2011\europarl-v7.lt-en.lt'

# Read the entire file as a string
with open(file_path, 'r', encoding='utf-8') as f:
    text = f.read()

# Replace spaces and tabs with newlines
text = text.replace(' ', '\n').replace('\t', '\n')

# Split by newlines to get words, strip whitespace, and filter out empty strings
words = [word.strip() for word in text.split('\n') if word.strip()]

# Use a set to collect unique words
all_words = set(words)

# Remove any empty strings if present (though strip should handle it)
all_words.discard("")

# Sort the words (optional, for readability)
sorted_words = sorted(all_words)

# Export to a text file (one word per line)
with open("data/European Parliament Proceedings Parallel Corpus 1996-2011/lit_words2.txt", "w", encoding="utf-8") as f:
    for word in sorted_words:
        f.write(word + "\n")

print(f"Exported {len(sorted_words)} unique words to lit_words.txt")

In [7]:
import os
from collections import Counter
import spacy

nlp = spacy.load("lt_core_news_sm", disable=["parser", "ner", "lemmatizer"])  # tokenization only
nlp.max_length = 2000000

base_path = r'data\\Lithuanian_Parliament_Corpus\\A-M_Lithuanian_Parliament_Corpus'
counter = Counter()

for folder_name in os.listdir(base_path):
    folder_path = os.path.join(base_path, folder_name)
    if os.path.isdir(folder_path):
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            if os.path.isfile(file_path):
                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    content = f.read()
                    # replace underscores with spaces (keeps prior behaviour)
                    content = content.replace('_', ' ')
                    lines = content.split('\n')
                    # original notebook used the 4th line (index 3) for the text — preserve that behaviour
                    if len(lines) >= 4:
                        text = lines[3].strip()
                        # tokenize and count
                        doc = nlp(text)
                        for token in doc:
                            counter[token.text] += 1

# Remove empty-string tokens if present
counter.pop("", None)

# Ensure output directory exists
out_dir = os.path.join('data', 'Lithuanian_Parliament_Corpus')
os.makedirs(out_dir, exist_ok=True)

# Export unique words (alphabetical) — preserves previous output
out_words = os.path.join(out_dir, 'unique_words.txt')
with open(out_words, 'w', encoding='utf-8') as f:
    for w in sorted(counter.keys()):
        f.write(w + '\n')

# Export counts as TSV (word<TAB>count), sorted by descending count
out_counts = os.path.join(out_dir, 'lithuanian_word_counts.tsv')
with open(out_counts, 'w', encoding='utf-8') as f:
    f.write('word\tcount\n')
    for w, c in counter.most_common():
        f.write(f"{w}\t{c}\n")

print(f"Exported {len(counter)} distinct words with counts to {out_counts}")
print(f"Exported {len(counter)} unique words to {out_words}")

Exported 326008 distinct words with counts to data\Lithuanian_Parliament_Corpus\lithuanian_word_counts.tsv
Exported 326008 unique words to data\Lithuanian_Parliament_Corpus\unique_words.txt


In [ ]:
# Read the file and extract only the words (first column)
file_path = r'data\en_full.txt'

words = []
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        # Split each line and take only the first part (the word)
        word = line.strip().split()[0]
        words.append(word)

# Filter for words with exactly 8 letters and use a set for uniqueness
unique_words = set(word for word in words if len(word) == 8)

# Export to a text file (one word per line)
with open("data/en_words_only.txt", "w", encoding="utf-8") as f:
    for word in unique_words:
        f.write(word + "\n")

In [ ]:
print(f"Extracted {len(unique_words)} unique words from en_full.txt")

In [ ]:
# Read the first file and extract words
with open(r'data\wiki-100k.txt', 'r', encoding='utf-8') as f:
    words1 = [line.strip() for line in f if line.strip()]

# Read the second file and extract words
with open(r'data\words_alpha.txt', 'r', encoding='utf-8') as f:
    words2 = [line.strip() for line in f if line.strip()]

# Combine and filter for exactly 8-letter words, using a set for uniqueness
unique_words_2 = set(word for word in words1 + words2 if len(word) == 10)

# Export to a text file (one word per line)
with open("8_en_words.txt", "w", encoding="utf-8") as f:
    for word in sorted(unique_words_2):
        f.write(word + "\n")

print(f"Exported {len(unique_words_2)} unique 8-letter words to 8_en_words.txt")

In [ ]:
# Combine the two sets
combined_words = unique_words.union(unique_words_2)

# Make all letters lowercase
combined_words_lower = {word.lower() for word in combined_words}

# Sort the words (optional, for readability)
sorted_words = sorted(combined_words_lower)

# Export to a text file (one word per line)
with open("combined_8_letter_words.txt", "w", encoding="utf-8") as f:
    for word in sorted_words:
        f.write(word + "\n")

print(f"Exported {len(sorted_words)} unique lowercase words to combined_8_letter_words.txt")

In [ ]:
# Read words from all_words_8_length.txt into a set
with open("all_words_8_length.txt", "r", encoding="utf-8") as f:
    words1 = set(line.strip() for line in f if line.strip())

# Read words from combined_8_letter_words.txt into a set
with open("combined_8_letter_words.txt", "r", encoding="utf-8") as f:
    words2 = set(line.strip() for line in f if line.strip())

# Find the intersection
common_words = words1.intersection(words2)

# Print the number of common words
print(f"Number of common words: {len(common_words)}")

In [ ]:
print(common_words)

In [ ]:
# Read words from all_words_8_length.txt into a set
with open("all_words_8_length.txt", "r", encoding="utf-8") as f:
    words = set(line.strip() for line in f if line.strip())

# Remove common words
filtered_words = words - common_words

# Sort the remaining words (optional, for consistency)
sorted_filtered_words = sorted(filtered_words)

# Write back to the file (overwriting) or to a new file
# To avoid overwriting, let's write to a new file
with open("all_words_8_length_filtered.txt", "w", encoding="utf-8") as f:
    for word in sorted_filtered_words:
        f.write(word + "\n")

print(f"Removed {len(common_words)} common words. Remaining words: {len(filtered_words)}")

In [ ]:
# Filter out words that start with two identical letters or end with the same two letters
new_filtered = [word for word in sorted_filtered_words if not (len(word) >= 4 and (word[0] == word[1] or word[-2] == word[-1]))]

# Update the list
sorted_filtered_words = new_filtered

print(f"After filtering, {len(sorted_filtered_words)} words remain.")

with open("all_words_8_length_filtered.txt", "w", encoding="utf-8") as f:
    for word in sorted_filtered_words:
        f.write(word + "\n")